# Native quantum Monte Carlo integration

Estimate the expectation of sin²(0.25x+0.2) for p(x)=x/28, x=0..7. Retain all three source programs: probability/amplitude preparation, the complete Grover operator, and three-bit signed fractional QPE. The compiler manages wires, composition and reversible work. The workflow computes source probabilities in full precision; the separately frozen Qmod counterpart retains its rounded decimal constants.


In [ ]:
from examples.synthesis.algorithms.amplitude_amplification_and_estimation.qmc_user_defined.overview.main import (
    build_program,
    plot_results,
    read_estimation,
)

probabilities = tuple(i / sum(range(8)) for i in range(8))
slope = 0.5
offset = 0.4
preparation = build_program("preparation", probabilities, slope, offset)
print("State preparation resources:", preparation.report["resources"])


## Build the Grover operator by composition

The quantum module implements probability loading and controlled Y rotations, the indicator phase oracle, the adjoint loader inside within/apply, the zero-state reflection and a constant global phase. Keep the full loader unitary, not only its first column: the inverse inside Grover depends on all columns. A seven-qubit budget holds four data/indicator qubits and three QPE qubits. No extra work qubits are used. The original cloud viewer is replaced by an actual native resource report.


In [ ]:
grover_program = build_program("grover", probabilities, slope, offset)
print("Grover resources:", grover_program.report["resources"])


## Three signed fractional phase bits and actual local sampling

Synthesize QPE around the same Grover callback. Dropped data remains physically retained; it is not zero workspace. Read phases using numeric_probabilities, preserving signed values with no manual bit decoding. Sample2048 outcomes with a fixed seed123 from the actual native distribution. This is explicit local classical sampling, not cloud or hardware execution. All eight bins, including zero counts, remain visible. Three phase bits set a resolution; they do not define a statistical confidence interval.


In [ ]:
from typing import Annotated

from examples.synthesis.algorithms.amplitude_amplification_and_estimation.qmc_user_defined.overview.quantum import (
    grover,
    load,
)
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, QNum, allocate, drop, qfunc, qpe


@qfunc
def integration_program(
    probabilities: list[float],
    angles: list[float],
    phase_result: Output[Annotated[QNum, (3, True, 3)]],
) -> None:
    state: QArray[QBit] = QArray()
    allocate(4, state)
    load(probabilities, angles, state)
    allocate(phase_result)
    qpe(lambda: grover(probabilities, angles, state), phase_result)
    drop(state)


estimation = synth.synthesize(
    integration_program,
    classical_inputs={
        "probabilities": list(probabilities),
        "angles": [slope * i + offset for i in range(8)],
    },
    max_qubits=7,
    max_workspace=0,
)
workflow = read_estimation(preparation, grover_program, estimation)
assert sum(count for _, count in workflow.counts) == 2048
print("Most frequent phase:", workflow.selected_phase)
print("Measured amplitude:", workflow.measured_amplitude)
print("Exact expectation:", workflow.exact_amplitude)
assert abs(workflow.measured_amplitude - workflow.exact_amplitude) < 0.1


## Compare the sampled phase peaks

Both signed peaks give the same sin²(pi*phase). The exact expectation is a classical validation calculation and never enters quantum state generation or phase selection. Complete complex states of all three stages are compared with independently assembled loading, reflection and Fourier matrices, including absolute phase and bit order. This small local example makes no hardware advantage or optimality claim.


In [ ]:
figure = plot_results(workflow)
